# SCD Type 2 (SCD2) — Implementation

## Purpose
This notebook contains the clean, reusable implementation of Slowly Changing Dimension Type 2 (SCD2).

SCD2 preserves historical versions of customer records when tracked attributes change.

## Processing flow
1. Validate the source data.
2. Validate the target table.
3. Identify new and changed customers.
4. Expire previous current versions when attributes change.
5. Insert new current versions.
6. Validate the resulting customer history.

## Table configuration
- Source table: `fmcg.gold.sb_dim_customers`
- Production target: `fmcg.gold.dim_customers`
- Practice target: `fmcg.gold.dim_customers_scd2_practice`
- Backup table: `fmcg.gold.dim_customers_backup`

## Safety rules
- Never modify the production or backup table during development and testing.
- Treat the original source table as read-only.
- Do not reset or overwrite the existing practice table.
- Test changes using dedicated test tables in the testing notebook.
- Do not execute SCD2 writes until source, target, and execution settings have been validated.
- Review the results and confirm exactly one current version per customer.
- Expiration and insertion are separate writes in the initial implementation; failure recovery must be considered.

## Execution policy
This notebook will be developed and validated step by step.
No data-changing operation should run until its target has been explicitly approved.


In [0]:

from pyspark.sql.utils import AnalysisException

# ============================================================
# 1. TABLE CONFIGURATION
# ============================================================

SOURCE_TABLE = "fmcg.gold.sb_dim_customers"

PRODUCTION_TARGET = "fmcg.gold.dim_customers"

PRACTICE_TARGET = "fmcg.gold.dim_customers_scd2_practice"

BACKUP_TABLE = "fmcg.gold.dim_customers_backup"

# The implementation must use the practice target during development.
TARGET_TABLE = PRACTICE_TARGET

# Primary key and columns tracked for changes
SOURCE_KEY = "customer_id"
TARGET_KEY = "customer_code"

TRACKED_COLUMNS = [
    "customer",
    "market",
    "platform",
    "channel"
]

# Keep execution disabled while building and validating the notebook.
EXECUTE_SCD2 = False


# ============================================================
# 2. READ-ONLY TABLE EXISTENCE CHECKS
# ============================================================

tables_to_check = {
    "Source": SOURCE_TABLE,
    "Practice target": PRACTICE_TARGET,
    "Production target": PRODUCTION_TARGET,
    "Backup": BACKUP_TABLE
}

print("SCD2 CONFIGURATION")
print("-" * 50)

for role, table_name in tables_to_check.items():
    try:
        exists = spark.catalog.tableExists(table_name)
        print(f"{role}: {table_name} | Exists: {exists}")
    except Exception as e:
        print(f"{role}: {table_name} | Check failed: {e}")


# ============================================================
# 3. BASIC CONFIGURATION SAFETY CHECKS
# ============================================================

assert TARGET_TABLE == PRACTICE_TARGET, (
    "Safety stop: TARGET_TABLE must point to the practice table."
)

assert TARGET_TABLE not in [PRODUCTION_TARGET, BACKUP_TABLE], (
    "Safety stop: production and backup tables cannot be targets."
)

assert len(TRACKED_COLUMNS) > 0, (
    "Safety stop: TRACKED_COLUMNS cannot be empty."
)

assert EXECUTE_SCD2 is False, (
    "Safety stop: execution must remain disabled during setup."
)

print("\nConfiguration checks passed.")
print(f"Configured target: {TARGET_TABLE}")
print(f"Execution enabled: {EXECUTE_SCD2}")


In [0]:

# ============================================================
# CELL 3: SCHEMA INSPECTION — READ ONLY
# ============================================================

source_df = spark.table(SOURCE_TABLE)
target_df = spark.table(TARGET_TABLE)

print("=" * 60)
print("SOURCE SCHEMA")
print("=" * 60)

source_df.printSchema()

print("\nSource columns:")
print(source_df.columns)


print("\n" + "=" * 60)
print("PRACTICE TARGET SCHEMA")
print("=" * 60)

target_df.printSchema()

print("\nTarget columns:")
print(target_df.columns)


# ============================================================
# REQUIRED COLUMN CHECKS
# ============================================================

required_source_columns = [
    SOURCE_KEY,
    *TRACKED_COLUMNS
]

required_target_columns = [
    TARGET_KEY,
    *TRACKED_COLUMNS,
    "effective_start_date",
    "effective_end_date",
    "is_current"
]

missing_source_columns = [
    col for col in required_source_columns
    if col not in source_df.columns
]

missing_target_columns = [
    col for col in required_target_columns
    if col not in target_df.columns
]

print("\n" + "=" * 60)
print("REQUIRED COLUMN VALIDATION")
print("=" * 60)

print("Missing source columns:", missing_source_columns)
print("Missing target columns:", missing_target_columns)

assert not missing_source_columns, (
    f"Source is missing required columns: {missing_source_columns}"
)

assert not missing_target_columns, (
    f"Practice target is missing required columns: {missing_target_columns}"
)

print("\nSchema validation passed.")


In [0]:

from pyspark.sql import functions as F

# ============================================================
# CELL 4: READ-ONLY DATA QUALITY VALIDATION
# ============================================================

source_df = spark.table(SOURCE_TABLE)
target_df = spark.table(TARGET_TABLE)

# ------------------------------------------------------------
# 1. SOURCE KEY VALIDATION
# ------------------------------------------------------------

source_null_keys = source_df.filter(
    F.col(SOURCE_KEY).isNull()
).count()

source_duplicate_keys = (
    source_df
    .filter(F.col(SOURCE_KEY).isNotNull())
    .groupBy(SOURCE_KEY)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# ------------------------------------------------------------
# 2. TARGET KEY AND SCD2 FIELD VALIDATION
# ------------------------------------------------------------

target_null_keys = target_df.filter(
    F.col(TARGET_KEY).isNull()
).count()

target_null_current_flags = target_df.filter(
    F.col("is_current").isNull()
).count()

target_null_start_dates = target_df.filter(
    F.col("effective_start_date").isNull()
).count()

# A current record must not have an end date.
invalid_current_dates = target_df.filter(
    (F.col("is_current") == True) &
    F.col("effective_end_date").isNotNull()
).count()

# A historical record must have an end date.
invalid_historical_dates = target_df.filter(
    (F.col("is_current") == False) &
    F.col("effective_end_date").isNull()
).count()

# An end date must not precede its start date.
invalid_date_order = target_df.filter(
    F.col("effective_end_date").isNotNull() &
    (F.col("effective_end_date") < F.col("effective_start_date"))
).count()

# ------------------------------------------------------------
# 3. CURRENT-VERSION VALIDATION
# ------------------------------------------------------------

multiple_current_customers = (
    target_df
    .filter(F.col("is_current") == True)
    .groupBy(TARGET_KEY)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Customers that have historical rows but no current row.
zero_current_customers = (
    target_df
    .groupBy(TARGET_KEY)
    .agg(
        F.sum(
            F.when(F.col("is_current") == True, 1).otherwise(0)
        ).alias("current_count")
    )
    .filter(F.col("current_count") == 0)
    .count()
)

# ------------------------------------------------------------
# 4. DISPLAY RESULTS
# ------------------------------------------------------------

validation_results = [
    ("Source rows", source_df.count()),
    ("Source null customer IDs", source_null_keys),
    ("Source duplicate customer IDs", source_duplicate_keys),
    ("Target rows", target_df.count()),
    ("Target null customer codes", target_null_keys),
    ("Target null current flags", target_null_current_flags),
    ("Target null start dates", target_null_start_dates),
    ("Current rows with end dates", invalid_current_dates),
    ("Historical rows without end dates", invalid_historical_dates),
    ("End dates before start dates", invalid_date_order),
    ("Customers with multiple current rows", multiple_current_customers),
    ("Customers with zero current rows", zero_current_customers),
]

print("=" * 65)
print("SCD2 DATA QUALITY REPORT — READ ONLY")
print("=" * 65)

for check_name, result in validation_results:
    print(f"{check_name}: {result}")

print("\nValidation report complete. No data was modified.")


In [0]:

from pyspark.sql import functions as F

# ============================================================
# CELL 5: SCD2 CHANGE CLASSIFICATION — READ ONLY
# ============================================================

source_df = spark.table(SOURCE_TABLE)
target_df = spark.table(TARGET_TABLE)

# Select only the current target versions.
current_target_df = (
    target_df
    .filter(F.col("is_current") == True)
    .select(
        F.col(TARGET_KEY).alias(SOURCE_KEY),
        *[F.col(c).alias(f"target_{c}") for c in TRACKED_COLUMNS]
    )
)

# Count all target versions per customer, including history.
target_customer_counts = (
    target_df
    .groupBy(F.col(TARGET_KEY).alias(SOURCE_KEY))
    .agg(
        F.count("*").alias("target_version_count"),
        F.sum(
            F.when(F.col("is_current") == True, 1).otherwise(0)
        ).alias("current_version_count")
    )
)

# Compare each source customer with its current target version.
comparison_df = (
    source_df
    .join(current_target_df, on=SOURCE_KEY, how="left")
    .join(target_customer_counts, on=SOURCE_KEY, how="left")
)

# Null-safe comparisons correctly handle null attribute values.
change_conditions = [
    ~F.col(c).eqNullSafe(F.col(f"target_{c}"))
    for c in TRACKED_COLUMNS
]

any_attribute_changed = change_conditions[0]

for condition in change_conditions[1:]:
    any_attribute_changed = any_attribute_changed | condition

comparison_df = comparison_df.withColumn(
    "scd2_action",
    F.when(
        F.coalesce(F.col("current_version_count"), F.lit(0)) > 1,
        F.lit("INVALID_TARGET")
    )
    .when(
        F.coalesce(F.col("current_version_count"), F.lit(0)) == 0,
        F.when(
            F.col("target_version_count").isNotNull(),
            F.lit("RECOVERY_REQUIRED")
        ).otherwise(F.lit("NEW"))
    )
    .when(any_attribute_changed, F.lit("CHANGED"))
    .otherwise(F.lit("UNCHANGED"))
)

# Display only the key and classification.
comparison_df.select(
    SOURCE_KEY,
    "scd2_action"
).orderBy(
    SOURCE_KEY
).show(100, truncate=False)

# Summary of expected actions.
print("SCD2 ACTION SUMMARY")

(
    comparison_df
    .groupBy("scd2_action")
    .count()
    .orderBy("scd2_action")
    .show(truncate=False)
)

# Safety: flag invalid current-version states before implementation.
invalid_count = comparison_df.filter(
    F.col("scd2_action") == "INVALID_TARGET"
).count()

assert invalid_count == 0, (
    f"Safety stop: {invalid_count} source customers have multiple current target versions."
)

print("Classification complete. No data was modified.")


In [0]:

# ============================================================
# CELL 6: REUSABLE SCD2 DRY-RUN PLAN
# READ ONLY — NO TARGET WRITES
# ============================================================

from pyspark.sql import functions as F


def build_scd2_plan(source_table, target_table):
    """
    Build a read-only SCD2 action plan.

    Possible actions:
      NEW
      CHANGED
      UNCHANGED
      RECOVERY_REQUIRED
      INVALID_TARGET

    This function does not modify any table.
    """

    source = spark.table(source_table)
    target = spark.table(target_table)

    # Validate required columns.
    required_source = [SOURCE_KEY, *TRACKED_COLUMNS]
    required_target = [
        TARGET_KEY,
        *TRACKED_COLUMNS,
        "effective_start_date",
        "effective_end_date",
        "is_current"
    ]

    missing_source = [c for c in required_source if c not in source.columns]
    missing_target = [c for c in required_target if c not in target.columns]

    if missing_source:
        raise ValueError(f"Missing source columns: {missing_source}")

    if missing_target:
        raise ValueError(f"Missing target columns: {missing_target}")

    # Validate source keys.
    if source.filter(F.col(SOURCE_KEY).isNull()).limit(1).count() > 0:
        raise ValueError("Source contains null customer IDs.")

    duplicate_source_keys = (
        source.groupBy(SOURCE_KEY)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicate_source_keys:
        raise ValueError("Source contains duplicate customer IDs.")

    # Validate target current-version counts.
    current_counts = (
        target.groupBy(TARGET_KEY)
        .agg(
            F.sum(
                F.when(F.col("is_current") == True, 1).otherwise(0)
            ).alias("current_count")
        )
    )

    if current_counts.filter(F.col("current_count") > 1).limit(1).count() > 0:
        raise ValueError("Target contains customers with multiple current versions.")

    # Join source to current target versions.
    current_target = (
        target.filter(F.col("is_current") == True)
        .select(
            F.col(TARGET_KEY).alias(SOURCE_KEY),
            *[F.col(c).alias(f"target_{c}") for c in TRACKED_COLUMNS]
        )
    )

    target_history = (
        target.groupBy(F.col(TARGET_KEY).alias(SOURCE_KEY))
        .agg(
            F.count("*").alias("target_version_count"),
            F.sum(
                F.when(F.col("is_current") == True, 1).otherwise(0)
            ).alias("current_version_count")
        )
    )

    joined = (
        source.join(current_target, SOURCE_KEY, "left")
        .join(target_history, SOURCE_KEY, "left")
    )

    # Null-safe comparison of tracked attributes.
    change_conditions = [
        ~F.col(c).eqNullSafe(F.col(f"target_{c}"))
        for c in TRACKED_COLUMNS
    ]

    any_changed = change_conditions[0]
    for condition in change_conditions[1:]:
        any_changed = any_changed | condition

    plan = joined.withColumn(
        "scd2_action",
        F.when(
            F.coalesce(F.col("current_version_count"), F.lit(0)) == 0,
            F.when(
                F.col("target_version_count").isNotNull(),
                F.lit("RECOVERY_REQUIRED")
            ).otherwise(F.lit("NEW"))
        )
        .when(any_changed, F.lit("CHANGED"))
        .otherwise(F.lit("UNCHANGED"))
    )

    # Source keys with multiple current target versions are rejected above.
    # Historical target keys absent from the source are reported separately.
    target_only = (
        target.select(F.col(TARGET_KEY).alias(SOURCE_KEY))
        .distinct()
        .join(source.select(SOURCE_KEY).distinct(), SOURCE_KEY, "left_anti")
    )

    summary = (
        plan.groupBy("scd2_action")
        .count()
        .orderBy("scd2_action")
    )

    return plan, summary, target_only


# ------------------------------------------------------------
# BUILD AND DISPLAY THE DRY-RUN PLAN
# ------------------------------------------------------------

scd2_plan_df, scd2_summary_df, target_only_df = build_scd2_plan(
    SOURCE_TABLE,
    TARGET_TABLE
)

print("DRY-RUN ACTION SUMMARY")
scd2_summary_df.show(truncate=False)

print("TARGET CUSTOMERS NOT PRESENT IN SOURCE")
target_only_df.show(100, truncate=False)

print("Dry-run plan built. No data was modified.")


In [0]:

# ============================================================
# CELL 7: INSPECT TARGET-ONLY CUSTOMER HISTORY
# READ ONLY
# ============================================================

target_only_keys = [
    row[SOURCE_KEY]
    for row in target_only_df.select(SOURCE_KEY).collect()
]

target_only_history_df = (
    spark.table(TARGET_TABLE)
    .filter(F.col(TARGET_KEY).isin(target_only_keys))
    .select(
        TARGET_KEY,
        *TRACKED_COLUMNS,
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .orderBy(TARGET_KEY, "effective_start_date")
)

print(f"Target-only customer keys: {len(target_only_keys)}")

print("\nVersion count by customer:")
(
    target_only_history_df
    .groupBy(TARGET_KEY)
    .agg(
        F.count("*").alias("version_count"),
        F.sum(
            F.when(F.col("is_current") == True, 1).otherwise(0)
        ).alias("current_versions")
    )
    .orderBy(TARGET_KEY)
    .show(100, truncate=False)
)

print("\nFull version history:")
target_only_history_df.show(200, truncate=False)

print("Inspection complete. No data was modified.")


# SCD2 Write Strategy and Safety Rules

## Source-to-target behavior

- NEW: Insert the first current version for a customer.
- CHANGED: Expire the existing current version and insert a new current version.
- UNCHANGED: Do not write a new version.
- RECOVERY_REQUIRED: Stop for investigation unless an explicit recovery procedure has been approved.
- INVALID_TARGET: Stop if a customer has multiple current versions.

## Target-only customers

Customers present in the target but absent from the source are preserved.
They are not automatically expired or deleted.

This rule assumes that the source represents the records to process, rather than a formally defined full snapshot that authorizes customer retirement.

## Write safeguards

- Production target and backup remain protected.
- The practice target is used for implementation development.
- Validate source keys and target history before writing.
- Build and review the dry-run plan before execution.
- Verify one current version per customer after processing.
- Do not automatically repair invalid history.
- Treat expiration and insertion as separate transactions unless a safer atomic design is implemented.
- Do not enable execution until the write logic has been reviewed and tested.


In [0]:

# ============================================================
# CELL 9: PREPARE SCD2 WRITE SETS
# READ ONLY — NO TARGET WRITES
# ============================================================

from pyspark.sql import functions as F

# Rebuild the plan from the latest source and target state.
scd2_plan_df, scd2_summary_df, target_only_df = build_scd2_plan(
    SOURCE_TABLE,
    TARGET_TABLE
)

# ------------------------------------------------------------
# 1. STOP IF RECOVERY OR INVALID STATES ARE FOUND
# ------------------------------------------------------------

recovery_count = scd2_plan_df.filter(
    F.col("scd2_action") == "RECOVERY_REQUIRED"
).count()

invalid_count = scd2_plan_df.filter(
    F.col("scd2_action") == "INVALID_TARGET"
).count()

if recovery_count > 0 or invalid_count > 0:
    raise ValueError(
        f"Safety stop: recovery_required={recovery_count}, "
        f"invalid_target={invalid_count}. Investigate before processing."
    )

# ------------------------------------------------------------
# 2. PREPARE ROWS TO EXPIRE
# ------------------------------------------------------------

rows_to_expire_df = (
    scd2_plan_df
    .filter(F.col("scd2_action") == "CHANGED")
    .select(SOURCE_KEY)
    .distinct()
)

# ------------------------------------------------------------
# 3. PREPARE NEW CURRENT VERSIONS
# ------------------------------------------------------------

# NEW and CHANGED source records both need a current target version.
rows_to_insert_df = (
    scd2_plan_df
    .filter(F.col("scd2_action").isin("NEW", "CHANGED"))
    .select(
        F.col(SOURCE_KEY).alias(TARGET_KEY),
        *TRACKED_COLUMNS
    )
    .withColumn("effective_start_date", F.current_timestamp())
    .withColumn(
        "effective_end_date",
        F.lit(None).cast("timestamp")
    )
    .withColumn("is_current", F.lit(True))
)

# ------------------------------------------------------------
# 4. REPORT THE PROPOSED WORK
# ------------------------------------------------------------

print("ROWS TO EXPIRE")
rows_to_expire_df.show(100, truncate=False)

print("ROWS TO INSERT")
rows_to_insert_df.show(100, truncate=False)

print("PROPOSED WRITE COUNTS")
print("Customers to expire:", rows_to_expire_df.count())
print("Versions to insert:", rows_to_insert_df.count())

print("\nACTION SUMMARY")
scd2_summary_df.show(truncate=False)

print("\nTarget-only customers preserved:", target_only_df.count())

print("\nPreparation complete. No data was modified.")


In [0]:

# ============================================================
# CELL 10: SCD2 WRITE-SAFETY GATE
# NO DATA WRITES
# ============================================================

def validate_scd2_execution(
    execute,
    target_table,
    plan_df,
    expected_target
):
    """
    Validate whether SCD2 execution is permitted.

    This function only validates settings and the dry-run plan.
    It does not modify data.
    """

    # Require an explicit execution request.
    if execute is not True:
        raise ValueError(
            "Execution is disabled. Set execute=True only after reviewing the plan."
        )

    # Require the exact approved development target.
    if target_table != expected_target:
        raise ValueError(
            f"Safety stop: expected target {expected_target}, "
            f"received {target_table}."
        )

    # Never permit production or backup targets in this notebook.
    if target_table in [PRODUCTION_TARGET, BACKUP_TABLE]:
        raise ValueError(
            "Safety stop: production and backup tables are protected."
        )

    # Ensure the plan does not contain unresolved recovery states.
    recovery_count = plan_df.filter(
        F.col("scd2_action") == "RECOVERY_REQUIRED"
    ).count()

    invalid_count = plan_df.filter(
        F.col("scd2_action") == "INVALID_TARGET"
    ).count()

    if recovery_count > 0 or invalid_count > 0:
        raise ValueError(
            f"Safety stop: recovery_required={recovery_count}, "
            f"invalid_target={invalid_count}."
        )

    return True


print("SCD2 safety gate defined.")
print("No data was modified.")


In [0]:

# ============================================================
# CELL 11: EXECUTION PREVIEW
# READ ONLY — NO TARGET WRITES
# ============================================================

def preview_scd2_execution():
    """
    Rebuild and validate the current SCD2 plan.
    Returns proposed actions without writing to any table.
    """

    # Rebuild the plan to avoid relying on stale DataFrames.
    plan_df, summary_df, target_only_df = build_scd2_plan(
        SOURCE_TABLE,
        TARGET_TABLE
    )

    # Check for unresolved states before preparing writes.
    recovery_count = plan_df.filter(
        F.col("scd2_action") == "RECOVERY_REQUIRED"
    ).count()

    invalid_count = plan_df.filter(
        F.col("scd2_action") == "INVALID_TARGET"
    ).count()

    if recovery_count or invalid_count:
        raise ValueError(
            f"Preview stopped: recovery_required={recovery_count}, "
            f"invalid_target={invalid_count}."
        )

    # Prepare the customer keys that would be expired.
    expire_df = (
        plan_df
        .filter(F.col("scd2_action") == "CHANGED")
        .select(SOURCE_KEY)
        .distinct()
    )

    # Prepare the new versions that would be inserted.
    insert_df = (
        plan_df
        .filter(F.col("scd2_action").isin("NEW", "CHANGED"))
        .select(
            F.col(SOURCE_KEY).alias(TARGET_KEY),
            *TRACKED_COLUMNS
        )
    )

    print("CURRENT ACTION SUMMARY")
    summary_df.show(truncate=False)

    print("CUSTOMERS PROPOSED FOR EXPIRATION")
    expire_df.show(100, truncate=False)

    print("SOURCE RECORDS PROPOSED FOR INSERTION")
    insert_df.show(100, truncate=False)

    print("Target-only customers preserved:", target_only_df.count())
    print("Proposed expirations:", expire_df.count())
    print("Proposed inserts:", insert_df.count())

    print("Preview complete. No data was modified.")

    return plan_df, expire_df, insert_df


# Execution remains disabled.
EXECUTE_SCD2 = False

print("Execution preview function defined.")
print("Execution enabled:", EXECUTE_SCD2)


In [0]:

# ============================================================
# CELL 12: RUN SCD2 PREVIEW
# READ ONLY — NO TARGET WRITES
# ============================================================

preview_plan_df, preview_expire_df, preview_insert_df = (
    preview_scd2_execution()
)


In [0]:

# ============================================================
# CELL 13: PRE-WRITE STATE VALIDATION
# READ ONLY — NO TARGET WRITES
# ============================================================

def validate_scd2_prewrite_state():
    # Restrict this notebook to the approved practice target.
    if TARGET_TABLE != PRACTICE_TARGET:
        raise ValueError(
            f"Safety stop: target must be {PRACTICE_TARGET}."
        )

    if TARGET_TABLE in [PRODUCTION_TARGET, BACKUP_TABLE]:
        raise ValueError("Safety stop: protected target selected.")

    # Rebuild the plan from the current source and target.
    plan_df, summary_df, target_only_df = build_scd2_plan(
        SOURCE_TABLE,
        TARGET_TABLE
    )

    # Reject unresolved recovery conditions.
    recovery_count = plan_df.filter(
        F.col("scd2_action") == "RECOVERY_REQUIRED"
    ).count()

    invalid_count = plan_df.filter(
        F.col("scd2_action") == "INVALID_TARGET"
    ).count()

    if recovery_count > 0 or invalid_count > 0:
        raise ValueError(
            f"Pre-write validation failed: recovery={recovery_count}, "
            f"invalid={invalid_count}."
        )

    # Validate target current-version counts.
    target_df = spark.table(TARGET_TABLE)

    target_version_state = (
        target_df
        .groupBy(TARGET_KEY)
        .agg(
            F.count("*").alias("version_count"),
            F.sum(
                F.when(F.col("is_current") == True, 1).otherwise(0)
            ).alias("current_count")
        )
    )

    multiple_current = target_version_state.filter(
        F.col("current_count") > 1
    ).count()

    zero_current = target_version_state.filter(
        F.col("current_count") == 0
    ).count()

    if multiple_current > 0 or zero_current > 0:
        raise ValueError(
            f"Target history invalid: multiple_current={multiple_current}, "
            f"zero_current={zero_current}."
        )

    print("Pre-write state validation passed.")
    print("Target:", TARGET_TABLE)
    print("Source rows:", spark.table(SOURCE_TABLE).count())
    print("Target rows:", target_df.count())
    print("Customers with multiple current versions:", multiple_current)
    print("Customers with zero current versions:", zero_current)
    print("Recovery cases:", recovery_count)
    print("Invalid target cases:", invalid_count)
    print("Target-only customers preserved:", target_only_df.count())

    return plan_df, summary_df, target_only_df


print("Pre-write validation function defined.")
print("No data was modified.")


In [0]:

# ============================================================
# CELL 14: RUN PRE-WRITE VALIDATION
# READ ONLY — NO TARGET WRITES
# ============================================================

validated_plan_df, validated_summary_df, validated_target_only_df = (
    validate_scd2_prewrite_state()
)

print("\nLATEST ACTION SUMMARY")
validated_summary_df.show(truncate=False)



# SCD2 Failure-Recovery Strategy

## The risk
The basic SCD2 process uses two separate writes:
1. Expire the old current version.
2. Insert the replacement current version.

If step 1 succeeds but step 2 fails, the customer may have no current version.

## Required safeguards

1. Validate the source and target before starting.
2. Identify all changed and new customers from a consistent input snapshot.
3. Record the intended run and its status so interrupted runs can be identified.
4. Expire old versions only for customers classified as CHANGED.
5. Insert exactly one replacement current version for each changed or new customer.
6. Validate the final state before marking the run successful.
7. If a failure occurs, stop further writes and identify which step completed.
8. Retry only after checking the actual target state; never blindly repeat an insert.
9. Keep target-only customers unchanged unless an approved business rule says otherwise.

## Important limitation

Delta Lake transactions are atomic at the individual table-operation level, but two separate operations are not automatically one atomic SCD2 transaction.

A production-grade solution must handle retries, concurrent runs, and failures between operations. Depending on the design, this may require a run-control table, idempotent merge logic, and a controlled recovery procedure.

## Current execution policy

The practice target remains the only permitted development target.
No writes are authorized until the implementation and recovery tests have passed.



# SCD2 Implementation Design

## Chosen approach
Use a controlled two-phase Delta Lake write process for the practice table.

### Phase 1 — Expire changed versions
Expire only the existing current versions identified as CHANGED.

### Phase 2 — Insert current versions
Insert source records classified as NEW or CHANGED, with:
- `effective_start_date` set for the new version
- `effective_end_date` set to NULL
- `is_current` set to TRUE

### Validation
Before writing:
- Validate source keys and target history.
- Rebuild the action plan.
- Reject unresolved recovery conditions.
- Confirm the practice target is selected.

After writing:
- Confirm every target customer has exactly one current version.
- Confirm current rows have no end date.
- Confirm historical rows have an end date.
- Confirm the source and target remain consistent for tracked attributes.

### Failure handling
The two phases are separate Delta transactions. If a failure occurs between them, stop processing and inspect the target state before retrying. Never blindly rerun the insert phase.

### Safety
- Production and backup tables remain protected.
- Execution stays disabled during development.
- No automatic expiration or deletion of target-only customers.


In [0]:

# ============================================================
# CELL 17: DEFINE THE EXPIRATION PHASE
# DEFINITION ONLY — NO WRITES
# ============================================================

def expire_changed_versions(
    changed_customer_keys,
    target_table,
    expiration_timestamp
):
    """
    Expire current target versions for changed customer keys.

    This function is intentionally not executed in this cell.
    """

    if target_table != PRACTICE_TARGET:
        raise ValueError(
            f"Safety stop: expiration is restricted to {PRACTICE_TARGET}."
        )

    if target_table in [PRODUCTION_TARGET, BACKUP_TABLE]:
        raise ValueError(
            "Safety stop: production and backup tables are protected."
        )

    if not changed_customer_keys:
        print("No changed customers. Nothing to expire.")
        return

    print(
        f"Expiration function defined for {len(changed_customer_keys)} "
        "customer keys."
    )
    print("No target changes were executed.")



In [0]:

# ============================================================
# CELL 18: DEFINE THE INSERTION PHASE
# DEFINITION ONLY — NO WRITES
# ============================================================

def insert_new_versions(
    rows_to_insert_df,
    target_table
):
    """
    Validate the insertion request without writing any rows.

    The actual insert operation will be implemented only after
    the expiration and recovery logic have been reviewed.
    """

    if target_table != PRACTICE_TARGET:
        raise ValueError(
            f"Safety stop: insertion is restricted to {PRACTICE_TARGET}."
        )

    if target_table in [PRODUCTION_TARGET, BACKUP_TABLE]:
        raise ValueError(
            "Safety stop: production and backup tables are protected."
        )

    required_columns = [
        TARGET_KEY,
        *TRACKED_COLUMNS,
        "effective_start_date",
        "effective_end_date",
        "is_current"
    ]

    missing_columns = [
        column for column in required_columns
        if column not in rows_to_insert_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Insertion DataFrame is missing columns: {missing_columns}"
        )

    print("Insertion function defined and its schema requirements validated.")
    print("No rows were inserted.")


print("Insertion function definition is ready.")
print("Execution remains disabled:", EXECUTE_SCD2)


In [0]:

# ============================================================
# CELL 19: DEFINE POST-RUN VALIDATION
# DEFINITION ONLY — NO WRITES
# ============================================================

def validate_scd2_target(target_table):
    """
    Validate SCD2 target invariants.
    Raises an error if any invariant fails.
    """

    if target_table != PRACTICE_TARGET:
        raise ValueError(
            f"Safety stop: validation is restricted to {PRACTICE_TARGET}."
        )

    target = spark.table(target_table)

    # Count current versions for each customer.
    customer_state = (
        target
        .groupBy(TARGET_KEY)
        .agg(
            F.count("*").alias("version_count"),
            F.sum(
                F.when(F.col("is_current") == True, 1).otherwise(0)
            ).alias("current_count")
        )
    )

    null_keys = target.filter(
        F.col(TARGET_KEY).isNull()
    ).count()

    null_current_flags = target.filter(
        F.col("is_current").isNull()
    ).count()

    null_start_dates = target.filter(
        F.col("effective_start_date").isNull()
    ).count()

    multiple_current = customer_state.filter(
        F.col("current_count") > 1
    ).count()

    zero_current = customer_state.filter(
        F.col("current_count") == 0
    ).count()

    invalid_current_dates = target.filter(
        (F.col("is_current") == True) &
        F.col("effective_end_date").isNotNull()
    ).count()

    invalid_historical_dates = target.filter(
        (F.col("is_current") == False) &
        F.col("effective_end_date").isNull()
    ).count()

    invalid_date_order = target.filter(
        F.col("effective_end_date").isNotNull() &
        (F.col("effective_end_date") < F.col("effective_start_date"))
    ).count()

    results = {
        "target_rows": target.count(),
        "null_keys": null_keys,
        "null_current_flags": null_current_flags,
        "null_start_dates": null_start_dates,
        "multiple_current_customers": multiple_current,
        "zero_current_customers": zero_current,
        "current_rows_with_end_dates": invalid_current_dates,
        "historical_rows_without_end_dates": invalid_historical_dates,
        "end_dates_before_start_dates": invalid_date_order
    }

    print("SCD2 POST-RUN VALIDATION")
    for name, value in results.items():
        print(f"{name}: {value}")

    failures = {
        name: value
        for name, value in results.items()
        if name != "target_rows" and value != 0
    }

    if failures:
        raise ValueError(f"SCD2 target validation failed: {failures}")

    print("All target-level SCD2 invariants passed.")
    return results


print("Post-run validation function defined.")
print("No data was modified.")


In [0]:

# ============================================================
# CELL 20: FINAL READ-ONLY CHECK BEFORE IMPLEMENTATION
# ============================================================

assert TARGET_TABLE == PRACTICE_TARGET
assert TARGET_TABLE not in [PRODUCTION_TARGET, BACKUP_TABLE]
assert EXECUTE_SCD2 is False

detail = spark.sql(
    f"DESCRIBE DETAIL {TARGET_TABLE}"
).first()

print("Target table:", TARGET_TABLE)
print("Table format:", detail["format"])
print("Location:", detail["location"])

target_before = spark.table(TARGET_TABLE)

print("Rows before implementation:", target_before.count())

target_before.groupBy("is_current").count().show()

validate_scd2_target(TARGET_TABLE)

print("Safety check complete. No data was modified.")


In [0]:

# ============================================================
# CELL 21: CREATE CONTROLLED SCD2 TEST INPUT
# READ-ONLY TABLES — TEMPORARY VIEW ONLY
# ============================================================

from pyspark.sql import functions as F

assert TARGET_TABLE == PRACTICE_TARGET
assert EXECUTE_SCD2 is False

source_df = spark.table(SOURCE_TABLE)

# Pick one existing customer for the CHANGED scenario.
test_customer = (
    source_df
    .select(SOURCE_KEY)
    .where(F.col(SOURCE_KEY).isNotNull())
    .orderBy(SOURCE_KEY)
    .first()[SOURCE_KEY]
)

# Simulate a change to a tracked attribute.
changed_test_df = (
    source_df
    .withColumn(
        "market",
        F.when(
            F.col(SOURCE_KEY) == test_customer,
            F.concat(F.col("market"), F.lit("_SCD2_TEST"))
        ).otherwise(F.col("market"))
    )
)

# Create one synthetic NEW customer using the source schema.
new_customer_values = {
    SOURCE_KEY: "SCD2_TEST_NEW_001",
    "customer_name": "SCD2 Test Customer",
    "city": "Test City",
    "customer": "SCD2 Test Customer",
    "market": "SCD2_TEST_MARKET",
    "platform": "SCD2_TEST_PLATFORM",
    "channel": "SCD2_TEST_CHANNEL"
}

new_customer_df = spark.createDataFrame(
    [new_customer_values],
    schema=source_df.schema
)

test_source_df = changed_test_df.unionByName(new_customer_df)

test_source_df.createOrReplaceTempView("scd2_controlled_test_source")

print("Existing customer selected for change:", test_customer)
print("Synthetic new customer:", new_customer_values[SOURCE_KEY])
print("Test source rows:", test_source_df.count())
print("Temporary view created: scd2_controlled_test_source")
print("No Delta table was modified.")


In [0]:

# ============================================================
# CELL 22: BUILD CONTROLLED TEST PLAN
# READ-ONLY — NO WRITES
# ============================================================

assert TARGET_TABLE == PRACTICE_TARGET
assert EXECUTE_SCD2 is False

test_plan_df, test_summary_df, test_target_only_df = build_scd2_plan(
    source_table="scd2_controlled_test_source",
    target_table=TARGET_TABLE
)

print("Test action summary:")
test_summary_df.show(truncate=False)

print("Expected changed/new customer classifications:")
test_plan_df.filter(
    F.col("scd2_action").isin("CHANGED", "NEW")
).select(
    SOURCE_KEY,
    "scd2_action"
).show(truncate=False)

print("Target-only customers remain preserved:")
print("Target-only key count:", test_target_only_df.count())

assert test_plan_df.filter(
    F.col("scd2_action") == "CHANGED"
).count() == 1

assert test_plan_df.filter(
    F.col("scd2_action") == "NEW"
).count() == 1

assert test_plan_df.filter(
    F.col("scd2_action").isin("RECOVERY_REQUIRED", "INVALID_TARGET")
).count() == 0

print("Controlled test plan passed. No data was modified.")


In [0]:

# ============================================================
# CELL 23: PREPARE CONTROLLED TEST WRITE SETS
# READ-ONLY — NO WRITES
# ============================================================

from pyspark.sql import functions as F

assert TARGET_TABLE == PRACTICE_TARGET
assert EXECUTE_SCD2 is False

# Keys whose current versions would need expiration.
keys_to_expire_df = (
    test_plan_df
    .filter(F.col("scd2_action") == "CHANGED")
    .select(F.col(SOURCE_KEY).alias(TARGET_KEY))
    .distinct()
)

# Source records requiring a new current version.
source_for_insert_df = (
    spark.table("scd2_controlled_test_source")
    .join(
        test_plan_df
        .filter(F.col("scd2_action").isin("NEW", "CHANGED"))
        .select(SOURCE_KEY),
        on=SOURCE_KEY,
        how="inner"
    )
)

# Map source columns to the target schema.
rows_to_insert_df = source_for_insert_df.select(
    F.col(SOURCE_KEY).alias(TARGET_KEY),
    *[F.col(c) for c in TRACKED_COLUMNS],
    F.current_timestamp().alias("effective_start_date"),
    F.lit(None).cast("timestamp").alias("effective_end_date"),
    F.lit(True).alias("is_current")
)

print("Keys to expire:")
keys_to_expire_df.show(truncate=False)

print("Rows prepared for insertion:")
rows_to_insert_df.show(truncate=False)

print("Expiration key count:", keys_to_expire_df.count())
print("Insertion row count:", rows_to_insert_df.count())
print("No Delta table was modified.")



In [0]:

# ============================================================
# CELL 24: PREPARE SINGLE-MERGE SCD2 STAGING DATA
# READ-ONLY — NO WRITES
# ============================================================

assert TARGET_TABLE == PRACTICE_TARGET
assert EXECUTE_SCD2 is False

# A single shared timestamp for this test batch.
batch_timestamp = rows_to_insert_df.select(
    F.max("effective_start_date").alias("ts")
).first()["ts"]

target_columns = [
    TARGET_KEY,
    *TRACKED_COLUMNS,
    "effective_start_date",
    "effective_end_date",
    "is_current"
]

# Stage 1: match existing current versions to expire.
expire_stage_df = (
    rows_to_insert_df
    .join(keys_to_expire_df, on=TARGET_KEY, how="inner")
    .select(
        F.col(TARGET_KEY).alias("merge_key"),
        *[F.col(c) for c in target_columns],
        F.lit("EXPIRE").alias("stage_action")
    )
    .withColumn("effective_start_date", F.lit(batch_timestamp))
)

# Stage 2: unmatched rows will be inserted as new current versions.
insert_stage_df = rows_to_insert_df.select(
    F.lit(None).cast("string").alias("merge_key"),
    *[F.col(c) for c in target_columns],
    F.lit("INSERT").alias("stage_action")
)

scd2_merge_stage_df = expire_stage_df.unionByName(insert_stage_df)

print("MERGE staging actions:")
scd2_merge_stage_df.groupBy("stage_action").count().show()

print("Staged records:")
scd2_merge_stage_df.select(
    "merge_key",
    TARGET_KEY,
    "market",
    "effective_start_date",
    "stage_action"
).show(truncate=False)

assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "EXPIRE"
).count() == 1

assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "INSERT"
).count() == 2

print("MERGE staging checks passed. No Delta table was modified.")



In [0]:

# ============================================================
# CELL 25: PREPARE CONSISTENT MERGE STAGING DATA
# READ-ONLY — NO WRITES
# ============================================================

assert TARGET_TABLE == PRACTICE_TARGET
assert EXECUTE_SCD2 is False

from pyspark.sql import functions as F

# Generate one timestamp for the entire batch.
batch_timestamp = spark.sql(
    "SELECT current_timestamp() AS ts"
).first()["ts"]

# Use the same timestamp in every staged row.
rows_to_insert_batch_df = rows_to_insert_df.withColumn(
    "effective_start_date",
    F.lit(batch_timestamp).cast("timestamp")
)

target_columns = [
    TARGET_KEY,
    *TRACKED_COLUMNS,
    "effective_start_date",
    "effective_end_date",
    "is_current"
]

# Existing current version to expire.
expire_stage_df = (
    rows_to_insert_batch_df
    .join(keys_to_expire_df, on=TARGET_KEY, how="inner")
    .select(
        F.col(TARGET_KEY).alias("merge_key"),
        *[F.col(c) for c in target_columns],
        F.lit("EXPIRE").alias("stage_action")
    )
)

# Rows to insert as new current versions.
insert_stage_df = rows_to_insert_batch_df.select(
    F.lit(None).cast("string").alias("merge_key"),
    *[F.col(c) for c in target_columns],
    F.lit("INSERT").alias("stage_action")
)

scd2_merge_stage_df = expire_stage_df.unionByName(insert_stage_df)

print("Batch timestamp:", batch_timestamp)

scd2_merge_stage_df.select(
    "merge_key",
    TARGET_KEY,
    "effective_start_date",
    "stage_action"
).show(truncate=False)

# Verify the single timestamp and expected operations.
assert scd2_merge_stage_df.select(
    "effective_start_date"
).distinct().count() == 1

assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "EXPIRE"
).count() == 1

assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "INSERT"
).count() == 2

print("Staging validation passed. No Delta table was modified.")


In [0]:

# ============================================================
# CELL 26: DEFINE GUARDED SCD2 DELTA MERGE
# DEFINITION ONLY — NO WRITES
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

def execute_scd2_merge(
    stage_df,
    target_table,
    execute=False
):
    """
    Apply SCD2 expiration and insertion in one Delta MERGE.

    Execution requires an explicit execute=True argument and
    is restricted to the practice target.
    """

    if execute is not True:
        raise ValueError(
            "Execution blocked. Explicit execute=True is required."
        )

    if target_table != PRACTICE_TARGET:
        raise ValueError(
            f"Safety stop: writes are restricted to {PRACTICE_TARGET}."
        )

    if target_table in [PRODUCTION_TARGET, BACKUP_TABLE]:
        raise ValueError(
            "Safety stop: production and backup tables are protected."
        )

    required_stage_columns = [
        "merge_key",
        *[
            TARGET_KEY,
            *TRACKED_COLUMNS,
            "effective_start_date",
            "effective_end_date",
            "is_current",
            "stage_action"
        ]
    ]

    missing = [
        c for c in required_stage_columns
        if c not in stage_df.columns
    ]
    if missing:
        raise ValueError(f"Missing staging columns: {missing}")

    allowed_actions = {"EXPIRE", "INSERT"}
    actions = {
        row["stage_action"]
        for row in stage_df.select("stage_action").distinct().collect()
    }

    if not actions.issubset(allowed_actions):
        raise ValueError(f"Unexpected staging actions: {actions}")

    expire_count = stage_df.filter(
        F.col("stage_action") == "EXPIRE"
    ).count()

    insert_count = stage_df.filter(
        F.col("stage_action") == "INSERT"
    ).count()

    if expire_count == 0 and insert_count == 0:
        print("No staged SCD2 changes. Nothing to write.")
        return

    # Each expiration must match one existing current version.
    expire_keys = (
        stage_df
        .filter(F.col("stage_action") == "EXPIRE")
        .select("merge_key")
    )

    if expire_keys.filter(F.col("merge_key").isNull()).count() > 0:
        raise ValueError("Expiration keys cannot be null.")

    if expire_keys.distinct().count() != expire_count:
        raise ValueError("Duplicate expiration keys found.")

    target_df = spark.table(target_table)

    missing_current = (
        expire_keys
        .join(
            target_df.filter(F.col("is_current") == True)
                .select(F.col(TARGET_KEY).alias("merge_key")),
            on="merge_key",
            how="left_anti"
        )
        .count()
    )

    if missing_current:
        raise ValueError(
            "An expiration key has no matching current target version."
        )

    # Insertion rows must use null merge keys to avoid matching
    # existing target versions.
    invalid_insert_keys = stage_df.filter(
        (F.col("stage_action") == "INSERT") &
        F.col("merge_key").isNotNull()
    ).count()

    if invalid_insert_keys:
        raise ValueError(
            "INSERT staging rows must have null merge_key values."
        )

    target_delta = DeltaTable.forName(spark, target_table)

    merge_condition = (
        f"t.{TARGET_KEY} = s.merge_key AND t.is_current = true"
    )

    update_values = {
        "effective_end_date": "s.effective_start_date",
        "is_current": "false"
    }

    insert_values = {
        TARGET_KEY: f"s.{TARGET_KEY}",
        **{c: f"s.{c}" for c in TRACKED_COLUMNS},
        "effective_start_date": "s.effective_start_date",
        "effective_end_date": "s.effective_end_date",
        "is_current": "s.is_current"
    }

    (
        target_delta.alias("t")
        .merge(stage_df.alias("s"), merge_condition)
        .whenMatchedUpdate(
            condition="s.stage_action = 'EXPIRE'",
            set=update_values
        )
        .whenNotMatchedInsert(
            condition="s.stage_action = 'INSERT'",
            values=insert_values
        )
        .execute()
    )

    print("Delta MERGE completed.")
    print("Expiration rows staged:", expire_count)
    print("Insertion rows staged:", insert_count)


print("Guarded SCD2 MERGE function defined.")
print("Execution remains disabled:", EXECUTE_SCD2)
print("No data was modified.")


In [0]:

# ============================================================
# CELL 27: VALIDATE MERGE PRECONDITIONS
# READ-ONLY — NO WRITES
# ============================================================

assert TARGET_TABLE == PRACTICE_TARGET
assert EXECUTE_SCD2 is False

# Confirm the practice target is structurally valid.
validate_scd2_target(TARGET_TABLE)

target_before_merge = spark.table(TARGET_TABLE)

# The changed customer must currently have exactly one current version.
changed_key = "789101"

changed_current_count = target_before_merge.filter(
    (F.col(TARGET_KEY) == changed_key) &
    (F.col("is_current") == True)
).count()

assert changed_current_count == 1, (
    f"Expected one current version for {changed_key}, "
    f"found {changed_current_count}"
)

# The synthetic new customer must not already exist in the target.
new_key = "SCD2_TEST_NEW_001"

new_key_count = target_before_merge.filter(
    F.col(TARGET_KEY) == new_key
).count()

assert new_key_count == 0, (
    f"Test key {new_key} already exists in the target."
)

# Confirm the staged rows have the expected actions.
assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "EXPIRE"
).count() == 1

assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "INSERT"
).count() == 2

print("Target rows before merge:", target_before_merge.count())
print("Current version count for changed customer:", changed_current_count)
print("Existing rows for synthetic new customer:", new_key_count)
print("Expiration rows staged: 1")
print("Insertion rows staged: 2")
print("Merge preconditions passed. No data was modified.")


In [0]:

# ============================================================
# CELL 28: EXECUTE CONTROLLED SCD2 MERGE
# WRITES TO PRACTICE TARGET ONLY
# ============================================================

assert TARGET_TABLE == PRACTICE_TARGET
assert EXECUTE_SCD2 is False

# Recheck the target immediately before writing.
validate_scd2_target(TARGET_TABLE)

target_now = spark.table(TARGET_TABLE)

assert target_now.filter(
    (F.col(TARGET_KEY) == "789101") &
    (F.col("is_current") == True)
).count() == 1, "Expected one current version for 789101."

assert target_now.filter(
    F.col(TARGET_KEY) == "SCD2_TEST_NEW_001"
).count() == 0, "Test customer already exists; stop to avoid duplicates."

assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "EXPIRE"
).count() == 1

assert scd2_merge_stage_df.filter(
    F.col("stage_action") == "INSERT"
).count() == 2

print("Executing one Delta MERGE on:", TARGET_TABLE)

execute_scd2_merge(
    stage_df=scd2_merge_stage_df,
    target_table=TARGET_TABLE,
    execute=True
)

print("MERGE call returned. Validating resulting target...")

validate_scd2_target(TARGET_TABLE)

print("Customer 789101 versions:")
spark.table(TARGET_TABLE).filter(
    F.col(TARGET_KEY) == "789101"
).orderBy("effective_start_date").show(truncate=False)

print("Synthetic customer:")
spark.table(TARGET_TABLE).filter(
    F.col(TARGET_KEY) == "SCD2_TEST_NEW_001"
).show(truncate=False)

print("Final row count:", spark.table(TARGET_TABLE).count())
print("Cell 28 completed.")


In [0]:

# ============================================================
# CELL 29: FINAL SCD2 TEST VERIFICATION
# READ-ONLY — NO WRITES
# ============================================================

final_target_df = spark.table(PRACTICE_TARGET)

print("Final current/history distribution:")
final_target_df.groupBy("is_current").count().show()

print("Versions for changed customer:")
final_target_df.filter(
    F.col(TARGET_KEY) == "789101"
).groupBy("is_current").count().show()

print("Synthetic new customer count:")
print(
    final_target_df.filter(
        F.col(TARGET_KEY) == "SCD2_TEST_NEW_001"
    ).count()
)

assert final_target_df.count() == 59
assert final_target_df.filter(
    (F.col(TARGET_KEY) == "789101") &
    (F.col("is_current") == True)
).count() == 1
assert final_target_df.filter(
    (F.col(TARGET_KEY) == "789101") &
    (F.col("is_current") == False)
).count() == 3
assert final_target_df.filter(
    (F.col(TARGET_KEY) == "SCD2_TEST_NEW_001") &
    (F.col("is_current") == True)
).count() == 1

validate_scd2_target(PRACTICE_TARGET)

print("All controlled SCD2 test assertions passed.")
print("Production and backup tables were not targeted.")
